# ============================================================
# LIMPIEZA DATASET GES
# Proyecto: Clasificación de diagnósticos GES
# Archivo entrada: dataset_original.csv
# Archivo salida: dataset_limpio.csv
# ============================================================


In [1]:
import pandas as pd
import unicodedata
import re


# ============================================================
# 1. Cargar dataset original
# ============================================================

In [2]:
df = pd.read_csv("../data/dataset_original.csv")

print("===== DATASET ORIGINAL =====")
print("Filas y columnas:", df.shape)
print(df.head())

print("\nInformación general:")
print(df.info())

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nDistribución original de GES:")
print(df["ges"].value_counts())


===== DATASET ORIGINAL =====
Filas y columnas: (941, 4)
    id                                       diagnostic  age    ges
0    1  RAQUIESTENOSIS L4-L5  Y DISCOPATIA SEVERA L5-S1   67  False
1   10                                  ABSCESO OVARICO   50  False
2  100                               Cálculo del uréter   57  False
3  101                               Cálculo del uréter   28  False
4  102                               Cálculo del uréter   54  False

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 941 entries, 0 to 940
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          941 non-null    int64 
 1   diagnostic  941 non-null    object
 2   age         941 non-null    int64 
 3   ges         941 non-null    bool  
dtypes: bool(1), int64(2), object(1)
memory usage: 23.1+ KB
None

Valores nulos por columna:
id            0
diagnostic    0
age           0
ges           0
dtype: int64


# ============================================================
# 2. Función para limpiar texto
# ============================================================


In [3]:
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto)

    # Pasar todo a mayúsculas
    texto = texto.upper()

    # Quitar tildes
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")

    # Reemplazar caracteres poco útiles por espacios
    texto = re.sub(r"[^A-Z0-9\s/+\-.]", " ", texto)

    # Quitar espacios repetidos
    texto = re.sub(r"\s+", " ", texto)

    # Quitar espacios al inicio y al final
    texto = texto.strip()

    return texto

# ============================================================
# 3. Limpiar columna diagnostic
# ============================================================


In [4]:
df["diagnostic"] = df["diagnostic"].apply(limpiar_texto)


# ============================================================
# 4. Limpiar columna age
# ============================================================

In [5]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# Edades inválidas
df.loc[df["age"] < 0, "age"] = None
df.loc[df["age"] > 110, "age"] = None

# En este dataset aparecen edades 0.
# Se consideran como dato faltante o mal ingresado.
df.loc[df["age"] == 0, "age"] = None

# Rellenar edades faltantes con la mediana
mediana_edad = df["age"].median()
df["age"] = df["age"].fillna(mediana_edad)

# Pasar edad a número entero
df["age"] = df["age"].astype(int)



# ============================================================
# 5. Limpiar columna ges
# ============================================================


In [6]:
df["ges"] = df["ges"].astype(str).str.strip()

df["ges"] = df["ges"].replace({
    "True": True,
    "False": False,
    "TRUE": True,
    "FALSE": False,
    "true": True,
    "false": False,
    "1": True,
    "0": False
})

df["ges"] = df["ges"].astype(bool)


C:\Users\JOSE\AppData\Local\Temp\ipykernel_2400\3491303973.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["ges"] = df["ges"].replace({


# ============================================================
# 6. Eliminar registros problemáticos
# ============================================================

In [7]:
filas_antes = len(df)

# Eliminar filas sin diagnóstico
df = df[df["diagnostic"] != ""]

# Eliminar duplicados exactos
df = df.drop_duplicates()

filas_despues = len(df)


# ============================================================
# 7. Revisión final
# ============================================================

In [8]:
print("\n===== DATASET LIMPIO =====")
print("Filas y columnas:", df.shape)
print(df.head())

print("\nFilas antes de limpieza:", filas_antes)
print("Filas después de limpieza:", filas_despues)
print("Filas eliminadas:", filas_antes - filas_despues)

print("\nValores nulos finales:")
print(df.isnull().sum())

print("\nDistribución final de GES:")
print(df["ges"].value_counts())

print("\nEdad mínima:", df["age"].min())
print("Edad máxima:", df["age"].max())
print("Edad promedio:", round(df["age"].mean(), 2))
print("Mediana de edad utilizada:", mediana_edad)

print("\nEjemplos de diagnósticos limpios:")
print(df["diagnostic"].head(20))



===== DATASET LIMPIO =====
Filas y columnas: (941, 4)
    id                                      diagnostic  age    ges
0    1  RAQUIESTENOSIS L4-L5 Y DISCOPATIA SEVERA L5-S1   67  False
1   10                                 ABSCESO OVARICO   50  False
2  100                              CALCULO DEL URETER   57  False
3  101                              CALCULO DEL URETER   28  False
4  102                              CALCULO DEL URETER   54  False

Filas antes de limpieza: 941
Filas después de limpieza: 941
Filas eliminadas: 0

Valores nulos finales:
id            0
diagnostic    0
age           0
ges           0
dtype: int64

Distribución final de GES:
ges
False    681
True     260
Name: count, dtype: int64

Edad mínima: 1
Edad máxima: 99
Edad promedio: 50.87
Mediana de edad utilizada: 56.0

Ejemplos de diagnósticos limpios:
0     RAQUIESTENOSIS L4-L5 Y DISCOPATIA SEVERA L5-S1
1                                    ABSCESO OVARICO
2                                 CALCULO DEL URETE

# ============================================================
# 8. Guardar dataset limpio
# ============================================================


In [9]:

df.to_csv("../data/dataset_limpio.csv", index=False, encoding="utf-8")

print("\nArchivo generado correctamente: dataset_limpio.csv")


Archivo generado correctamente: dataset_limpio.csv
